In [73]:
import pandas as pd
import numpy as np

In [74]:
optimized = pd.read_csv(
    "../data/processed/final_slot_recommendations.csv"
)

warehouse_slots = pd.read_csv(
    "../data/processed/warehouse_slots.csv"
)

print("Optimized data shape:", optimized.shape)
print("Warehouse slots shape:", warehouse_slots.shape)

print("\nOptimized columns:")
print(optimized.columns.tolist())

print("\nWarehouse columns:")
print(warehouse_slots.columns.tolist())

Optimized data shape: (3922, 11)
Warehouse slots shape: (25, 3)

Optimized columns:
['StockCode', 'order_frequency', 'total_quantity', 'avg_quantity_per_order', 'related_product_count', 'cluster_name', 'recommended_zone', 'distance_from_packing', 'picking_priority', 'current_distance', 'distance_saved']

Warehouse columns:
['slot', 'x', 'y']


In [75]:
optimized["StockCode"] = (
    optimized["StockCode"]
    .astype(str)
    .str.strip()
)

warehouse_slots["slot"] = (
    warehouse_slots["slot"]
    .astype(str)
    .str.strip()
)

In [76]:
zone_mapping = {
    "A": "Zone A",
    "B": "Zone B",
    "C": "Zone C",
    "D": "Zone D",
    "E": "Zone E"
}

warehouse_slots["zone"] = (
    warehouse_slots["slot"]
    .str[0]
    .map(zone_mapping)
)

In [77]:
print("Recommended zones:")
print(
    optimized["recommended_zone"]
    .value_counts(dropna=False)
)

print("\nWarehouse zones:")
print(
    warehouse_slots["zone"]
    .value_counts(dropna=False)
)

Recommended zones:
recommended_zone
Zone B    2192
Zone D    1551
Zone A     178
NaN          1
Name: count, dtype: int64

Warehouse zones:
zone
Zone A    5
Zone B    5
Zone C    5
Zone D    5
Zone E    5
Name: count, dtype: int64


In [78]:
optimized["recommended_zone"] = (
    optimized["recommended_zone"]
    .fillna("Zone C")
)

In [79]:
optimized["picking_priority"] = pd.to_numeric(
    optimized["picking_priority"],
    errors="coerce"
)

optimized["picking_priority"] = (
    optimized["picking_priority"]
    .fillna(0)
)

In [80]:
optimized = optimized.sort_values(
    by="picking_priority",
    ascending=False
).reset_index(drop=True)

In [81]:
print(
    optimized[
        [
            "StockCode",
            "cluster_name",
            "recommended_zone",
            "picking_priority"
        ]
    ].head(20)
)

   StockCode          cluster_name recommended_zone  picking_priority
0     85099B  High-Volume Products           Zone A         101047019
1     85123A  High-Volume Products           Zone A          82734918
2      22197  High-Volume Products           Zone A          79202016
3      84879  High-Volume Products           Zone A          52906710
4      21212  High-Volume Products           Zone A          48042720
5      47566  High-Volume Products           Zone A          30806855
6      23084  High-Volume Products           Zone A          30554566
7      20725  High-Volume Products           Zone A          30411080
8      84077  High-Volume Products           Zone A          29398785
9      22423  High-Volume Products           Zone A          27535788
10     22386  High-Volume Products           Zone A          26123664
11     23203  High-Volume Products           Zone A          25217035
12     22178  High-Volume Products           Zone A          25173175
13     22086  High-V

In [82]:
def assign_slots_by_zone(products, warehouse):

    products = products.copy()

    # Create empty column
    products["optimized_slot"] = None

    # Process each recommended zone
    for zone in products["recommended_zone"].unique():

        # Products belonging to this zone
        product_indices = products[
            products["recommended_zone"] == zone
        ].index.tolist()

        # Slots available in this zone
        zone_slots = warehouse[
            warehouse["zone"] == zone
        ].sort_values(
            ["x", "y"]
        )

        available_slots = (
            zone_slots["slot"]
            .tolist()
        )

        # If no slots exist for this zone
        if len(available_slots) == 0:

            print(
                f"WARNING: No slots available for {zone}"
            )

            continue

        # Assign products to available slots
        for position, product_index in enumerate(
            product_indices
        ):

            slot_index = (
                position % len(available_slots)
            )

            products.loc[
                product_index,
                "optimized_slot"
            ] = available_slots[slot_index]

    return products

In [83]:
optimized = assign_slots_by_zone(
    optimized,
    warehouse_slots
)

In [84]:
missing_slots = optimized[
    optimized["optimized_slot"].isna()
]

print(
    "Total products:",
    len(optimized)
)

print(
    "Products with slots:",
    optimized["optimized_slot"].notna().sum()
)

print(
    "Missing slots:",
    optimized["optimized_slot"].isna().sum()
)

Total products: 3922
Products with slots: 3922
Missing slots: 0


In [85]:
print(
    optimized[
        [
            "StockCode",
            "cluster_name",
            "recommended_zone",
            "picking_priority",
            "optimized_slot"
        ]
    ].head(20)
)

   StockCode          cluster_name recommended_zone  picking_priority  \
0     85099B  High-Volume Products           Zone A         101047019   
1     85123A  High-Volume Products           Zone A          82734918   
2      22197  High-Volume Products           Zone A          79202016   
3      84879  High-Volume Products           Zone A          52906710   
4      21212  High-Volume Products           Zone A          48042720   
5      47566  High-Volume Products           Zone A          30806855   
6      23084  High-Volume Products           Zone A          30554566   
7      20725  High-Volume Products           Zone A          30411080   
8      84077  High-Volume Products           Zone A          29398785   
9      22423  High-Volume Products           Zone A          27535788   
10     22386  High-Volume Products           Zone A          26123664   
11     23203  High-Volume Products           Zone A          25217035   
12     22178  High-Volume Products           Zone A

In [86]:
warehouse_slots["slot_distance"] = (
    abs(warehouse_slots["x"] - 0)
    +
    abs(warehouse_slots["y"] - 0)
)

In [87]:
optimized = optimized.merge(
    warehouse_slots[
        [
            "slot",
            "x",
            "y",
            "zone",
            "slot_distance"
        ]
    ],
    left_on="optimized_slot",
    right_on="slot",
    how="left"
)

In [88]:
optimized = optimized.rename(
    columns={
        "slot_distance": "optimized_distance",
        "x": "optimized_x",
        "y": "optimized_y"
    }
)

In [89]:
optimized = optimized.drop(
    columns=["slot"],
    errors="ignore"
)

In [90]:
optimized["current_distance"] = pd.to_numeric(
    optimized["current_distance"],
    errors="coerce"
)

In [91]:
optimized["actual_distance_saved"] = (
    optimized["current_distance"]
    -
    optimized["optimized_distance"]
)

In [92]:
print(
    optimized[
        [
            "StockCode",
            "current_distance",
            "optimized_slot",
            "optimized_distance",
            "actual_distance_saved"
        ]
    ].head(20)
)

   StockCode  current_distance optimized_slot  optimized_distance  \
0     85099B                60             A1                   0   
1     85123A                60             A2                   1   
2      22197                60             A3                   2   
3      84879                60             A4                   3   
4      21212                60             A5                   4   
5      47566                60             A1                   0   
6      23084                60             A2                   1   
7      20725                60             A3                   2   
8      84077                60             A4                   3   
9      22423                60             A5                   4   
10     22386                60             A1                   0   
11     23203                60             A2                   1   
12     22178                60             A3                   2   
13     22086                60    

In [93]:
final_columns = [
    "StockCode",
    "order_frequency",
    "total_quantity",
    "avg_quantity_per_order",
    "related_product_count",
    "cluster_name",
    "recommended_zone",
    "optimized_slot",
    "optimized_x",
    "optimized_y",
    "picking_priority",
    "current_distance",
    "optimized_distance",
    "actual_distance_saved"
]

In [94]:
final_slot_recommendations = optimized[
    [
        col
        for col in final_columns
        if col in optimized.columns
    ]
].copy()

In [95]:
print(
    final_slot_recommendations.head(20)
)

   StockCode  order_frequency  total_quantity  avg_quantity_per_order  \
0     85099B             2089           48371               22.935514   
1     85123A             2198           37641               16.707057   
2      22197             1392           56898               40.125529   
3      84879             1455           36362               24.635501   
4      21212             1320           36396               26.920118   
5      47566             1685           18283               10.761036   
6      23084              994           30739               30.225172   
7      20725             1565           19432               12.283186   
8      84077              535           54951              102.520522   
9      22423             1988           13851                6.901345   
10     22386             1218           21448               17.409091   
11     23203             1231           20485               16.493559   
12     22178             1037           24275      

In [96]:
print(
    "\nRows:",
    len(final_slot_recommendations)
)

print(
    "Missing StockCode:",
    final_slot_recommendations["StockCode"].isna().sum()
)

print(
    "Missing Zone:",
    final_slot_recommendations["recommended_zone"].isna().sum()
)

print(
    "Missing Slot:",
    final_slot_recommendations["optimized_slot"].isna().sum()
)

print(
    "Missing Optimized Distance:",
    final_slot_recommendations["optimized_distance"].isna().sum()
)


Rows: 3922
Missing StockCode: 0
Missing Zone: 0
Missing Slot: 0
Missing Optimized Distance: 0


In [98]:
final_slot_recommendations.to_csv(
    "../data/processed/final_slot_recommendations.csv",
    index=False
)